In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("emmarex/plantdisease")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'plantdisease' dataset.
Path to dataset files: /kaggle/input/plantdisease


In [2]:
import os, cv2, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import *
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
import tensorflow as tf

In [3]:
import os
print(os.listdir(path))

['PlantVillage', 'plantvillage']


In [4]:
dataset_path = os.path.join(path, os.listdir(path)[0])
print(dataset_path)
print(os.listdir(dataset_path)[:5])

/kaggle/input/plantdisease/PlantVillage
['Pepper__bell___Bacterial_spot', 'Potato___healthy', 'Tomato_Leaf_Mold', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato_Bacterial_spot']


In [ ]:
import cv2
import numpy as np

IMG_SIZE = 128

images = []
labels = []

for cls in os.listdir(dataset_path):

    cls_path = os.path.join(dataset_path, cls)

    if os.path.isdir(cls_path):

        for img_file in os.listdir(cls_path):

            img_path = os.path.join(cls_path, img_file)

            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            images.append(gray)
            labels.append(cls)

X = np.array(images)

print("Shape:", X.shape)

In [ ]:
sobel_x = np.array([[-1,0,1],
                    [-2,0,2],
                    [-1,0,1]])

sobel_y = np.array([[-1,-2,-1],
                    [0,0,0],
                    [1,2,1]])

lap = np.array([[0,-1,0],
                [-1,4,-1],
                [0,-1,0]])

processed_images = []

for img in X:

    sx = cv2.filter2D(img, -1, sobel_x)
    sy = cv2.filter2D(img, -1, sobel_y)

    sobel = np.sqrt(sx**2 + sy**2)

    lap_img = cv2.filter2D(img, -1, lap)

    merged = np.stack([img, sobel, lap_img], axis=-1)

    processed_images.append(merged)

X = np.array(processed_images) / 255.0
print(X.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

classes = len(set(labels))

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(labels)

y = to_categorical(y, classes)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

baseline = Sequential([
    Dense(512, activation='relu'),
    Dropout(0.4),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(classes, activation='softmax')
])

baseline.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

baseline.fit(X_train_flat, y_train, epochs=10, validation_split=0.2)

baseline.evaluate(X_test_flat, y_test)

In [ ]:
from tensorflow.keras.layers import *
from tensorflow.keras.models import Sequential

cnn = Sequential([

    Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
    BatchNormalization(),
    MaxPool2D(),

    Conv2D(64,(3,3),activation='relu'),
    BatchNormalization(),
    MaxPool2D(),

    Conv2D(128,(3,3),activation='relu'),
    BatchNormalization(),
    MaxPool2D(),

    Flatten(),
    Dense(256,activation='relu'),
    Dropout(0.5),
    Dense(classes,activation='softmax')
])

cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn.fit(X_train,y_train,epochs=15,validation_split=0.2)

cnn.evaluate(X_test,y_test)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

base = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(256,activation='relu')(x)
x = Dropout(0.4)(x)
out = Dense(classes,activation='softmax')(x)

model = Model(base.input, out)

model.compile(
    optimizer=Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train,y_train,epochs=10,validation_split=0.2)

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

base.trainable = True

for layer in base.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3
)

model.fit(
    X_train, y_train,
    epochs=10,
    validation_split=0.2,
    callbacks=[lr]
)

model.evaluate(X_test, y_test)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

base = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(256,activation='relu')(x)
x = Dropout(0.4)(x)
out = Dense(classes,activation='softmax')(x)

model = Model(base.input, out)

model.compile(
    optimizer=Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train,y_train,epochs=10,validation_split=0.2)

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

base.trainable = True

for layer in base.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3
)

model.fit(
    X_train, y_train,
    epochs=10,
    validation_split=0.2,
    callbacks=[lr]
)

model.evaluate(X_test, y_test)